In [1]:
import duckdb
con = duckdb.connect("../olist.duckdb", read_only=True)

In [2]:
con.execute("""
SELECT
    purchase_month,
    COUNT(DISTINCT customer_unique_id)                          AS customers,
    COUNT(*)                                                    AS orders,
    ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT customer_unique_id), 3) AS orders_per_cust,
    ROUND(AVG(order_revenue), 2)                                AS avg_order_value,
    ROUND(SUM(order_revenue), 0)                                AS revenue
FROM fact_orders
GROUP BY 1 ORDER BY 1
""").df()

,purchase_month,customers,orders,orders_per_cust,avg_order_value,revenue
0,2017-01-01,718,749,1.043,169.93,127276.0
1,2017-02-01,1628,1651,1.014,164.01,270783.0
2,2017-03-01,2507,2545,1.015,162.76,414217.0
3,2017-04-01,2274,2303,1.013,169.70,390812.0
4,2017-05-01,3477,3544,1.019,159.80,566337.0
5,2017-06-01,3075,3134,1.019,156.31,489871.0
6,2017-07-01,3791,3861,1.018,146.14,564236.0
7,2017-08-01,4112,4191,1.019,153.97,645294.0
8,2017-09-01,4083,4150,1.016,168.93,701077.0
9,2017-10-01,4417,4478,1.014,167.73,751117.0


In [3]:
con.execute("""
WITH m AS (
    SELECT purchase_month,
           COUNT(DISTINCT customer_unique_id) AS cust,
           COUNT(*) * 1.0 / COUNT(DISTINCT customer_unique_id) AS freq,
           AVG(order_revenue) AS aov
    FROM fact_orders GROUP BY 1
),
p AS (
    SELECT
        CASE WHEN purchase_month < DATE '2017-07-01' THEN 'H1_2017'
             WHEN purchase_month >= DATE '2018-03-01' THEN 'H1_2018'
        END AS period,
        AVG(cust) AS cust, AVG(freq) AS freq, AVG(aov) AS aov
    FROM m
    WHERE purchase_month < DATE '2017-07-01'
       OR purchase_month >= DATE '2018-03-01'
    GROUP BY 1
)
SELECT period,
       ROUND(cust, 0) AS avg_monthly_customers,
       ROUND(freq, 3)  AS orders_per_customer,
       ROUND(aov, 2)   AS avg_order_value,
       ROUND(cust * freq * aov, 0) AS implied_monthly_revenue
FROM p ORDER BY period
""").df()

,period,avg_monthly_customers,orders_per_customer,avg_order_value,implied_monthly_revenue
0,H1_2017,2280.0,1.021,163.75,381019.0
1,H1_2018,6469.0,1.009,163.65,1067810.0


In [4]:
import os
os.makedirs("../outputs", exist_ok=True)

monthly = con.execute("""
SELECT purchase_month,
       COUNT(DISTINCT customer_unique_id) AS customers,
       COUNT(*) AS orders,
       ROUND(COUNT(*)*1.0/COUNT(DISTINCT customer_unique_id),3) AS orders_per_cust,
       ROUND(AVG(order_revenue),2) AS aov,
       ROUND(SUM(order_revenue),0) AS revenue
FROM fact_orders GROUP BY 1 ORDER BY 1
""").df()

monthly.to_csv("../outputs/monthly_metrics.csv", index=False)
print("saved")

saved


In [5]:
con.execute("""
WITH first_order AS (
    SELECT customer_unique_id,
           MIN(order_purchase_timestamp) AS first_ts
    FROM fact_orders GROUP BY 1
),
cohort AS (
    SELECT f.customer_unique_id,
           date_trunc('month', f.first_ts)::DATE AS cohort_month,
           f.first_ts
    FROM first_order f
),
repeats AS (
    SELECT c.cohort_month,
           c.customer_unique_id,
           MIN(date_diff('day', c.first_ts::DATE, o.purchase_date)) AS days_to_second
    FROM cohort c
    JOIN fact_orders o
      ON o.customer_unique_id = c.customer_unique_id
     AND o.order_purchase_timestamp > c.first_ts
    GROUP BY 1, 2
)
SELECT
    c.cohort_month,
    COUNT(DISTINCT c.customer_unique_id) AS cohort_size,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN r.days_to_second <= 30
          THEN r.customer_unique_id END) / COUNT(DISTINCT c.customer_unique_id), 2) AS rep_30d,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN r.days_to_second <= 90
          THEN r.customer_unique_id END) / COUNT(DISTINCT c.customer_unique_id), 2) AS rep_90d,
    ROUND(100.0 * COUNT(DISTINCT r.customer_unique_id)
          / COUNT(DISTINCT c.customer_unique_id), 2) AS rep_ever
FROM cohort c
LEFT JOIN repeats r ON c.customer_unique_id = r.customer_unique_id
GROUP BY 1 ORDER BY 1
""").df()

,cohort_month,cohort_size,rep_30d,rep_90d,rep_ever
0,2017-01-01,718,2.65,2.92,6.55
1,2017-02-01,1626,1.29,1.60,3.94
2,2017-03-01,2502,1.52,2.24,4.60
3,2017-04-01,2257,1.24,1.86,4.25
4,2017-05-01,3449,1.80,2.49,4.99
5,2017-06-01,3036,1.68,2.50,5.01
6,2017-07-01,3743,1.66,2.16,4.27
7,2017-08-01,4056,1.55,2.34,4.31
8,2017-09-01,4005,1.62,2.50,4.17
9,2017-10-01,4328,1.36,1.99,3.44


In [6]:
con.execute("""
WITH first_order AS (
    SELECT customer_unique_id, MIN(order_purchase_timestamp) AS first_ts
    FROM fact_orders GROUP BY 1
),
cohort AS (
    SELECT customer_unique_id,
           date_trunc('month', first_ts)::DATE AS cohort_month,
           first_ts
    FROM first_order
),
repeats AS (
    SELECT c.customer_unique_id, c.cohort_month,
           MIN(date_diff('day', c.first_ts::DATE, o.purchase_date)) AS days_to_second
    FROM cohort c
    JOIN fact_orders o
      ON o.customer_unique_id = c.customer_unique_id
     AND o.order_purchase_timestamp > c.first_ts
    GROUP BY 1, 2
)
SELECT
    c.cohort_month,
    COUNT(DISTINCT c.customer_unique_id) AS cohort_size,
    CASE WHEN c.cohort_month <= DATE '2018-07-01' THEN
      ROUND(100.0*COUNT(DISTINCT CASE WHEN r.days_to_second<=30 THEN r.customer_unique_id END)
            /COUNT(DISTINCT c.customer_unique_id),2) END AS rep_30d,
    CASE WHEN c.cohort_month <= DATE '2018-05-01' THEN
      ROUND(100.0*COUNT(DISTINCT CASE WHEN r.days_to_second<=90 THEN r.customer_unique_id END)
            /COUNT(DISTINCT c.customer_unique_id),2) END AS rep_90d
FROM cohort c
LEFT JOIN repeats r ON c.customer_unique_id = r.customer_unique_id
GROUP BY 1 ORDER BY 1
""").df()

,cohort_month,cohort_size,rep_30d,rep_90d
0,2017-01-01,718,2.65,2.92
1,2017-02-01,1626,1.29,1.60
2,2017-03-01,2502,1.52,2.24
3,2017-04-01,2257,1.24,1.86
4,2017-05-01,3449,1.80,2.49
5,2017-06-01,3036,1.68,2.50
6,2017-07-01,3743,1.66,2.16
7,2017-08-01,4056,1.55,2.34
8,2017-09-01,4005,1.62,2.50
9,2017-10-01,4328,1.36,1.99


In [7]:
con.execute("""
SELECT
    is_late,
    COUNT(*)                       AS orders,
    ROUND(AVG(review_score), 2)    AS avg_review,
    ROUND(100.0 * AVG(CASE WHEN review_score = 1 THEN 1 ELSE 0 END), 2) AS pct_one_star,
    ROUND(100.0 * AVG(CASE WHEN review_score >= 4 THEN 1 ELSE 0 END), 2) AS pct_four_plus
FROM fact_orders
WHERE review_score IS NOT NULL
GROUP BY 1 ORDER BY 1
""").df()

,is_late,orders,avg_review,pct_one_star,pct_four_plus
0,False,89163,4.29,6.58,82.71
1,True,6378,2.27,53.70,26.78


In [8]:
con.execute("""
SELECT
    CASE WHEN days_vs_promise <= -10 THEN 'a. 10+ days early'
         WHEN days_vs_promise <   0  THEN 'b. 1-9 days early'
         WHEN days_vs_promise =   0  THEN 'c. on promise'
         WHEN days_vs_promise <=  3  THEN 'd. 1-3 days late'
         WHEN days_vs_promise <=  7  THEN 'e. 4-7 days late'
         WHEN days_vs_promise <= 14  THEN 'f. 8-14 days late'
         ELSE                             'g. 15+ days late' END AS bucket,
    COUNT(*)                    AS orders,
    ROUND(AVG(review_score), 2) AS avg_review,
    ROUND(100.0 * AVG(CASE WHEN review_score = 1 THEN 1 ELSE 0 END), 2) AS pct_one_star
FROM fact_orders
WHERE review_score IS NOT NULL
GROUP BY 1 ORDER BY 1
""").df()

,bucket,orders,avg_review,pct_one_star
0,a. 10+ days early,61252,4.33,6.37
1,b. 1-9 days early,26631,4.23,6.98
2,c. on promise,1280,4.04,8.44
3,d. 1-3 days late,1851,3.29,25.18
4,e. 4-7 days late,1748,2.11,58.47
5,f. 8-14 days late,1446,1.67,70.61
6,g. 15+ days late,1333,1.73,68.72


In [9]:
con.execute("""
WITH first_orders AS (
    SELECT customer_unique_id, order_id, purchase_date, is_late, review_score
    FROM fact_orders
    WHERE customer_order_seq = 1
      AND purchase_date <= DATE '2018-05-31'
),
returned AS (
    SELECT DISTINCT f.customer_unique_id
    FROM first_orders f
    JOIN fact_orders o
      ON o.customer_unique_id = f.customer_unique_id
     AND o.purchase_date >  f.purchase_date
     AND o.purchase_date <= f.purchase_date + INTERVAL 90 DAY
)
SELECT
    f.is_late,
    COUNT(*)                                                        AS customers,
    COUNT(r.customer_unique_id)                                     AS returned_90d,
    ROUND(100.0 * COUNT(r.customer_unique_id) / COUNT(*), 3)        AS repeat_rate_pct
FROM first_orders f
LEFT JOIN returned r ON f.customer_unique_id = r.customer_unique_id
GROUP BY 1 ORDER BY 1
""").df()

,is_late,customers,returned_90d,repeat_rate_pct
0,False,69420,916,1.320
1,True,5694,58,1.019


In [10]:
con.execute("""
WITH first_orders AS (
    SELECT customer_unique_id, purchase_date, review_score
    FROM fact_orders
    WHERE customer_order_seq = 1
      AND purchase_date <= DATE '2018-05-31'
      AND review_score IS NOT NULL
),
returned AS (
    SELECT DISTINCT f.customer_unique_id
    FROM first_orders f
    JOIN fact_orders o
      ON o.customer_unique_id = f.customer_unique_id
     AND o.purchase_date >  f.purchase_date
     AND o.purchase_date <= f.purchase_date + INTERVAL 90 DAY
)
SELECT
    f.review_score,
    COUNT(*)                                                 AS customers,
    ROUND(100.0 * COUNT(r.customer_unique_id) / COUNT(*), 3) AS repeat_rate_pct
FROM first_orders f
LEFT JOIN returned r ON f.customer_unique_id = r.customer_unique_id
GROUP BY 1 ORDER BY 1
""").df()

,review_score,customers,repeat_rate_pct
0,1,7736,1.008
1,2,2361,1.059
2,3,6416,1.091
3,4,14943,1.211
4,5,43128,1.428


In [11]:
from scipy.stats import chi2_contingency
import numpy as np

# on-time: 916 of 69420 | late: 58 of 5694
table = np.array([[916, 69420-916],
                  [58,  5694-58]])
chi2, p, dof, _ = chi2_contingency(table)
print(f"chi-square = {chi2:.3f}   p-value = {p:.4f}")

chi-square = 3.491   p-value = 0.0617


In [12]:
con.execute("""
WITH seller_perf AS (
    SELECT
        seller_id,
        COUNT(DISTINCT order_id)                                    AS orders,
        SUM(CASE WHEN is_late THEN 1 ELSE 0 END)                    AS late_orders,
        ROUND(100.0*AVG(CASE WHEN is_late THEN 1 ELSE 0 END), 2)    AS late_rate,
        ROUND(AVG(review_score), 2)                                 AS avg_review
    FROM fact_order_items
    WHERE seller_count = 1
    GROUP BY 1
)
SELECT
    COUNT(*)                              AS total_sellers,
    SUM(orders)                           AS total_orders,
    SUM(late_orders)                      AS total_late,
    ROUND(100.0*SUM(late_orders)/SUM(orders), 2) AS overall_late_rate
FROM seller_perf
""").df()

,total_sellers,total_orders,total_late,overall_late_rate
0,2925,94931.0,7232.0,7.62


In [13]:
con.execute("""
WITH seller_perf AS (
    SELECT seller_id,
           COUNT(DISTINCT order_id) AS orders,
           SUM(CASE WHEN is_late THEN 1 ELSE 0 END) AS late_orders
    FROM fact_order_items
    WHERE seller_count = 1
    GROUP BY 1
),
ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (ORDER BY late_orders DESC) AS rn,
           SUM(late_orders) OVER (ORDER BY late_orders DESC
                                  ROWS UNBOUNDED PRECEDING) AS cum_late,
           SUM(orders)      OVER (ORDER BY late_orders DESC
                                  ROWS UNBOUNDED PRECEDING) AS cum_orders,
           SUM(late_orders) OVER () AS all_late,
           SUM(orders)      OVER () AS all_orders,
           COUNT(*)         OVER () AS n_sellers
    FROM seller_perf
)
SELECT
    ROUND(100.0*rn/n_sellers, 1)          AS pct_of_sellers,
    ROUND(100.0*cum_orders/all_orders, 1) AS pct_of_orders,
    ROUND(100.0*cum_late/all_late, 1)     AS pct_of_late_deliveries
FROM ranked
WHERE rn IN (10, 25, 50, 100, 200, 300, 500)
ORDER BY rn
""").df()

,pct_of_sellers,pct_of_orders,pct_of_late_deliveries
0,0.3,13.8,15.9
1,0.9,22.6,27.6
2,1.7,31.6,38.0
3,3.4,41.4,50.4
4,6.8,54.8,65.3
5,10.3,62.8,74.2
6,17.1,73.0,84.8


In [14]:
con.execute("""
WITH seller_perf AS (
    SELECT seller_id,
           COUNT(DISTINCT order_id) AS orders,
           SUM(CASE WHEN is_late THEN 1 ELSE 0 END) AS late_orders,
           100.0*AVG(CASE WHEN is_late THEN 1 ELSE 0 END) AS late_rate,
           AVG(review_score) AS avg_review
    FROM fact_order_items
    WHERE seller_count = 1
    GROUP BY 1
    HAVING COUNT(DISTINCT order_id) >= 50
),
bucketed AS (
    SELECT *,
           NTILE(10) OVER (ORDER BY late_rate) AS decile
    FROM seller_perf
)
SELECT decile,
       COUNT(*)                               AS sellers,
       SUM(orders)                            AS orders,
       ROUND(100.0*SUM(orders)/SUM(SUM(orders)) OVER (), 1) AS pct_of_orders,
       SUM(late_orders)                       AS late_orders,
       ROUND(100.0*SUM(late_orders)/SUM(SUM(late_orders)) OVER (), 1) AS pct_of_late,
       ROUND(AVG(late_rate), 2)               AS avg_late_rate,
       ROUND(AVG(avg_review), 2)              AS avg_review
FROM bucketed
GROUP BY 1 ORDER BY 1
""").df()

,decile,sellers,orders,pct_of_orders,late_orders,pct_of_late,avg_late_rate,avg_review
0,1,42,3857.0,5.4,39.0,0.7,0.82,4.33
1,2,42,5291.0,7.4,145.0,2.7,2.48,4.28
2,3,41,4915.0,6.9,192.0,3.6,3.54,4.20
3,4,41,6492.0,9.1,311.0,5.8,4.31,4.24
4,5,41,12900.0,18.1,723.0,13.4,5.19,4.21
5,6,41,7495.0,10.5,541.0,10.0,6.25,4.17
6,7,41,7951.0,11.1,701.0,13.0,7.31,4.14
7,8,41,10189.0,14.3,1010.0,18.7,8.78,4.10
8,9,41,7396.0,10.4,891.0,16.5,10.86,3.89
9,10,41,4841.0,6.8,843.0,15.6,16.07,3.86


In [15]:
con.execute("""
WITH seller_perf AS (
    SELECT seller_id,
           COUNT(DISTINCT order_id) AS orders,
           SUM(CASE WHEN is_late THEN 1 ELSE 0 END) AS late_orders,
           100.0*AVG(CASE WHEN is_late THEN 1 ELSE 0 END) AS late_rate
    FROM fact_order_items WHERE seller_count = 1
    GROUP BY 1 HAVING COUNT(DISTINCT order_id) >= 50
),
bucketed AS (SELECT *, NTILE(10) OVER (ORDER BY late_rate) AS decile FROM seller_perf)
SELECT
    SUM(orders)                                    AS orders_in_bottom_2,
    SUM(late_orders)                               AS current_late,
    ROUND(SUM(orders) * 0.076, 0)                  AS late_if_platform_avg,
    ROUND(SUM(late_orders) - SUM(orders)*0.076, 0) AS avoidable_late_orders
FROM bucketed WHERE decile >= 9
""").df()

,orders_in_bottom_2,current_late,late_if_platform_avg,avoidable_late_orders
0,12237.0,1734.0,930.0,804.0


In [16]:
con.execute("""
SELECT
    ROUND(AVG(order_revenue), 2) AS avg_order_value,
    COUNT(*) AS late_orders,
    ROUND(SUM(order_revenue), 0) AS revenue_of_late_orders
FROM fact_orders WHERE is_late
""").df()

,avg_order_value,late_orders,revenue_of_late_orders
0,176.17,6531,1150550.0


In [17]:
con.execute("""
SELECT
    (SELECT COUNT(*) FROM fact_orders WHERE is_late) AS late_orders_fact,
    (SELECT COUNT(DISTINCT order_id) FROM fact_order_items
     WHERE is_late AND seller_count = 1)             AS late_single_seller,
    (SELECT COUNT(*) FROM fact_orders
     WHERE is_late AND seller_count > 1)             AS late_multi_seller
""").df()

,late_orders_fact,late_single_seller,late_multi_seller
0,6531,6518,13


In [18]:
con.execute("""
WITH sp AS (
    SELECT seller_id,
           COUNT(DISTINCT order_id) AS orders,
           100.0*AVG(CASE WHEN is_late THEN 1 ELSE 0 END) AS late_rate
    FROM fact_order_items WHERE seller_count = 1
    GROUP BY 1 HAVING COUNT(DISTINCT order_id) >= 50
),
b AS (SELECT *, NTILE(10) OVER (ORDER BY late_rate) AS decile FROM sp)
SELECT
    CASE WHEN b.decile >= 9 THEN 'bottom 2 deciles' ELSE 'rest' END AS grp,
    COUNT(DISTINCT i.order_id) AS orders,
    ROUND(AVG(i.review_score), 2) AS avg_review,
    ROUND(100.0*AVG(CASE WHEN i.review_score = 1 THEN 1 ELSE 0 END), 2) AS pct_one_star
FROM fact_order_items i
JOIN b ON i.seller_id = b.seller_id
WHERE i.seller_count = 1
GROUP BY 1 ORDER BY 1
""").df()

,grp,orders,avg_review,pct_one_star
0,bottom 2 deciles,12237,3.89,15.21
1,rest,59090,4.15,9.82


In [19]:
import os
os.makedirs("../outputs/dashboard", exist_ok=True)
P = "../outputs/dashboard/"

# 1. Monthly trend — executive page
con.execute("""
SELECT purchase_month,
       COUNT(DISTINCT customer_unique_id) AS customers,
       COUNT(*) AS orders,
       ROUND(SUM(order_revenue),0) AS revenue,
       ROUND(AVG(order_revenue),2) AS aov,
       ROUND(100.0*AVG(CASE WHEN is_late THEN 1 ELSE 0 END),2) AS late_rate,
       ROUND(AVG(review_score),2) AS avg_review
FROM fact_orders GROUP BY 1 ORDER BY 1
""").df().to_csv(P+"monthly_trend.csv", index=False)

# 2. Delivery buckets vs review
con.execute("""
SELECT CASE WHEN days_vs_promise <= -10 THEN '1. 10+ days early'
            WHEN days_vs_promise <   0  THEN '2. 1-9 days early'
            WHEN days_vs_promise =   0  THEN '3. On promise'
            WHEN days_vs_promise <=  3  THEN '4. 1-3 days late'
            WHEN days_vs_promise <=  7  THEN '5. 4-7 days late'
            WHEN days_vs_promise <= 14  THEN '6. 8-14 days late'
            ELSE '7. 15+ days late' END AS bucket,
       COUNT(*) AS orders,
       ROUND(AVG(review_score),2) AS avg_review,
       ROUND(100.0*AVG(CASE WHEN review_score=1 THEN 1 ELSE 0 END),2) AS pct_one_star
FROM fact_orders WHERE review_score IS NOT NULL
GROUP BY 1 ORDER BY 1
""").df().to_csv(P+"delivery_buckets.csv", index=False)

# 3. Seller scorecard — operational page
con.execute("""
WITH sp AS (
    SELECT i.seller_id,
           COUNT(DISTINCT i.order_id) AS orders,
           SUM(CASE WHEN i.is_late THEN 1 ELSE 0 END) AS late_orders,
           100.0*AVG(CASE WHEN i.is_late THEN 1 ELSE 0 END) AS late_rate,
           AVG(i.review_score) AS avg_review,
           SUM(i.price + i.freight_value) AS revenue
    FROM fact_order_items i WHERE i.seller_count = 1
    GROUP BY 1 HAVING COUNT(DISTINCT i.order_id) >= 50
)
SELECT sp.seller_id, d.seller_state,
       sp.orders, sp.late_orders,
       ROUND(sp.late_rate,2) AS late_rate,
       ROUND(sp.avg_review,2) AS avg_review,
       ROUND(sp.revenue,0) AS revenue,
       NTILE(10) OVER (ORDER BY sp.late_rate) AS late_decile,
       CASE WHEN NTILE(10) OVER (ORDER BY sp.late_rate) >= 9
            THEN 'Action required' ELSE 'OK' END AS flag
FROM sp LEFT JOIN dim_seller d ON sp.seller_id = d.seller_id
ORDER BY sp.late_rate DESC
""").df().to_csv(P+"seller_scorecard.csv", index=False)

# 4. State-level
con.execute("""
SELECT d.customer_state,
       COUNT(*) AS orders,
       ROUND(100.0*AVG(CASE WHEN f.is_late THEN 1 ELSE 0 END),2) AS late_rate,
       ROUND(AVG(f.review_score),2) AS avg_review,
       ROUND(AVG(f.delivery_days),1) AS avg_delivery_days
FROM fact_orders f JOIN dim_customer d USING (customer_unique_id)
GROUP BY 1 HAVING COUNT(*) >= 100 ORDER BY late_rate DESC
""").df().to_csv(P+"state_performance.csv", index=False)

# 5. Cohort retention (censored properly)
con.execute("""
WITH fo AS (SELECT customer_unique_id, MIN(order_purchase_timestamp) AS first_ts
            FROM fact_orders GROUP BY 1),
c AS (SELECT customer_unique_id, date_trunc('month', first_ts)::DATE AS cohort_month, first_ts FROM fo),
r AS (SELECT c.customer_unique_id, MIN(date_diff('day', c.first_ts::DATE, o.purchase_date)) AS d2
      FROM c JOIN fact_orders o ON o.customer_unique_id=c.customer_unique_id
      AND o.order_purchase_timestamp > c.first_ts GROUP BY 1)
SELECT c.cohort_month,
       COUNT(DISTINCT c.customer_unique_id) AS cohort_size,
       CASE WHEN c.cohort_month <= DATE '2018-07-01' THEN
         ROUND(100.0*COUNT(DISTINCT CASE WHEN r.d2<=30 THEN r.customer_unique_id END)
               /COUNT(DISTINCT c.customer_unique_id),2) END AS rep_30d,
       CASE WHEN c.cohort_month <= DATE '2018-05-01' THEN
         ROUND(100.0*COUNT(DISTINCT CASE WHEN r.d2<=90 THEN r.customer_unique_id END)
               /COUNT(DISTINCT c.customer_unique_id),2) END AS rep_90d
FROM c LEFT JOIN r USING (customer_unique_id)
GROUP BY 1 ORDER BY 1
""").df().to_csv(P+"cohort_retention.csv", index=False)

print("5 files exported to outputs/dashboard/")

5 files exported to outputs/dashboard/


In [1]:
state_names = """customer_state,state_name
AC,Acre
AL,Alagoas
AP,Amapá
AM,Amazonas
BA,Bahia
CE,Ceará
DF,Distrito Federal
ES,Espírito Santo
GO,Goiás
MA,Maranhão
MT,Mato Grosso
MS,Mato Grosso do Sul
MG,Minas Gerais
PA,Pará
PB,Paraíba
PR,Paraná
PE,Pernambuco
PI,Piauí
RJ,Rio de Janeiro
RN,Rio Grande do Norte
RS,Rio Grande do Sul
RO,Rondônia
RR,Roraima
SC,Santa Catarina
SP,São Paulo
SE,Sergipe
TO,Tocantins"""

with open("../outputs/dashboard/state_names.csv", "w", encoding="utf-8") as f:
    f.write(state_names)

import pandas as pd
sp = pd.read_csv("../outputs/dashboard/state_performance.csv")
sn = pd.read_csv("../outputs/dashboard/state_names.csv")
sp = sp.merge(sn, on="customer_state", how="left")
sp["state_label"] = sp["state_name"] + " (" + sp["customer_state"] + ")"
sp.to_csv("../outputs/dashboard/state_performance.csv", index=False)
print(sp[["customer_state","state_label"]].head())

  customer_state    state_label
0             AL   Alagoas (AL)
1             MA  Maranhão (MA)
2             SE   Sergipe (SE)
3             PI     Piauí (PI)
4             CE     Ceará (CE)


In [2]:
ss = pd.read_csv("../outputs/dashboard/seller_scorecard.csv")
sn2 = sn.rename(columns={"customer_state":"seller_state","state_name":"seller_state_name"})
ss = ss.merge(sn2, on="seller_state", how="left")
ss.to_csv("../outputs/dashboard/seller_scorecard.csv", index=False)
print("done")

done


In [1]:
import pandas as pd
ss = pd.read_csv("../outputs/dashboard/seller_scorecard.csv")
ss = ss.sort_values("late_rate", ascending=False)
ss.to_csv("../outputs/dashboard/seller_scorecard.csv", index=False)
print(ss[["seller_id","late_rate","flag"]].head())

                          seller_id  late_rate             flag
0  54965bbe3e4f07ae045b90b0b8541f52      32.50  Action required
1  bbad7e518d7af88a0897397ffdca1979      23.75  Action required
2  beadbee30901a7f61d031b6b686095ad      22.39  Action required
3  6039e27294dc75811c0d8a39069f52c0      21.92  Action required
4  a49928bcdf77c55c6d6e05e09a9b4ca5      21.78  Action required
